In [1]:
# --- Imports ---
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

# --- Define paths ---
project_root = Path("..")       # your project root (notebooks/ is inside project/)
data_dir = project_root / "data"
processed_dir = data_dir / "processed"

# Create processed folder if not exists
processed_dir.mkdir(exist_ok=True)

print("Folders ready:", processed_dir)


Folders ready: ..\data\processed


In [8]:
# Paths (already defined previously)
cup_tr_path = data_dir / "ML-CUP25-TR.csv"
cup_ts_path = data_dir / "ML-CUP25-TS.csv"

# Load training + blind test files:
# - engine="python" makes the parser more flexible for weird spacing
# - sep=r"\s+" means "one or more spaces/tabs"
# - comment="#" ignores any header/comment lines starting with '#'
# Load training + blind test CSVs (comma-separated, with comment lines ignored)
cup_tr = pd.read_csv(
    cup_tr_path,
    sep=",",
    header=None,
    comment="#"
)

cup_ts = pd.read_csv(
    cup_ts_path,
    sep=",",
    header=None,
    comment="#"
)

print("Training shape:", cup_tr.shape)
print("Blind test shape:", cup_ts.shape)

cup_tr.head()



Training shape: (500, 17)
Blind test shape: (1000, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,1,-6.925642,-6.093158,-9.149763,-5.918488,4.391259,-1.059304,-5.031085,-6.932177,-5.805652,7.147028,4.555533,-5.694865,6.554997,10.688732,15.416160,-7.535628
1,2,-5.649870,-7.650998,-10.407383,-7.864047,3.790306,-1.673732,-8.493233,-8.143588,-9.447557,10.790796,6.266211,-5.551301,12.342252,-8.135250,23.787661,-3.270978
2,3,15.985886,14.192953,24.466835,12.551305,-7.788409,0.557977,23.145951,20.031774,14.516358,-21.024198,-10.410913,12.061133,28.542661,-14.132383,-56.408372,1.892238
3,4,12.774004,10.156462,18.588934,8.346695,-5.245173,-0.199274,14.500231,12.608063,12.411055,-15.479452,-8.871887,6.703585,20.253500,9.525402,-0.673842,40.295464
4,5,-4.019226,-4.043457,-5.095354,-3.147125,0.725466,-0.477673,-4.025913,-0.995364,-3.491760,3.385533,1.838361,-4.271710,-3.588910,6.050010,1.198489,-11.677909


In [7]:
with open(cup_tr_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(10):
        print(repr(f.readline()))


'# Data Set: 4D-2025\n'
'# Nov 2025 - ML-2025 CUP\n'
'# INFO: micheli@di.unipi.it - domenico.tortorella@unipi.it - alessandro.dipalma@phd.unipi.it\n'
'# (C) CIML group - Micheli 2025\n'
'# Format:\n'
'# Training set: ID, INPUTS, TARGET_1, TARGET_2, TARGET_3, TARGET_4 (last 4 columns)\n'
'# Blind Test set: ID, INPUTS\n'
'1,-6.925642084298755,-6.093158480105726,-9.149763110667456,-5.9184880166644245,4.391259275705645,-1.0593044330712527,-5.0310854350046,-6.932176563713492,-5.805652370607629,7.147028426616375,4.5555330039202335,-5.694864773073439,6.55499658385967,10.6887318416127,15.416160435125,-7.53562784994093\n'
'2,-5.649869941831543,-7.650997694281324,-10.40738298464612,-7.864046610135294,3.7903062745050162,-1.6737321428591259,-8.493232601200596,-8.14358759492449,-9.447557352262223,10.790795894859468,6.266210652257294,-5.551300783050679,12.3422524206817,-8.13524963668272,23.7876608328669,-3.27097753012155\n'
'3,15.985885772639167,14.19295315601233,24.46683536308024,12.55130543595208,

In [9]:
# TRAINING SET
ids_tr = cup_tr.iloc[:, 0].to_numpy()

# X = columns 1..12
X = cup_tr.iloc[:, 1:13].to_numpy(dtype=float)

# y = last 4 columns (13,14,15,16)
y = cup_tr.iloc[:, 13:17].to_numpy(dtype=float)

# BLIND TEST SET
ids_ts = cup_ts.iloc[:, 0].to_numpy()
X_test_blind = cup_ts.iloc[:, 1:13].to_numpy(dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Blind test X shape:", X_test_blind.shape)


X shape: (500, 12)
y shape: (500, 4)
Blind test X shape: (1000, 12)


In [10]:
from sklearn.model_selection import train_test_split

# First split into: 80% temp + 20% internal-test
X_temp, X_test_int, y_temp, y_test_int = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Second split into: train (60%) + val (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, shuffle=True
)

print("TRAIN:", X_train.shape)
print("VAL:", X_val.shape)
print("INTERNAL TEST:", X_test_int.shape)


TRAIN: (300, 12)
VAL: (100, 12)
INTERNAL TEST: (100, 12)


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on TRAIN
X_train_scaled = scaler.fit_transform(X_train)

# Transform the others
X_val_scaled = scaler.transform(X_val)
X_test_int_scaled = scaler.transform(X_test_int)
X_test_blind_scaled = scaler.transform(X_test_blind)

print("Scaling complete.")


Scaling complete.


In [12]:
# Save raw splits
np.save(processed_dir / "X_train.npy", X_train)
np.save(processed_dir / "y_train.npy", y_train)
np.save(processed_dir / "X_val.npy", X_val)
np.save(processed_dir / "y_val.npy", y_val)
np.save(processed_dir / "X_test_int.npy", X_test_int)
np.save(processed_dir / "y_test_int.npy", y_test_int)

# Save scaled splits
np.save(processed_dir / "X_train_scaled.npy", X_train_scaled)
np.save(processed_dir / "X_val_scaled.npy", X_val_scaled)
np.save(processed_dir / "X_test_int_scaled.npy", X_test_int_scaled)
np.save(processed_dir / "X_test_blind_scaled.npy", X_test_blind_scaled)

# Save blind test IDs
np.save(processed_dir / "ids_ts.npy", ids_ts)

print("All processed data saved.")


All processed data saved.


In [13]:
joblib.dump(scaler, processed_dir / "scaler_cup.joblib")
print("Scaler saved.")


Scaler saved.


In [14]:
print("Mean train:", X_train_scaled.mean())
print("STD train:", X_train_scaled.std())
print("Any NaNs?", np.isnan(X_train_scaled).any())


Mean train: -5.921189464667502e-18
STD train: 0.9999999999999999
Any NaNs? False
